In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import os
import glob
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ---------- Locate data files anywhere under /kaggle/input or current folder ----------
def find_file(name):
    hits = (glob.glob(f"/kaggle/input/**/{name}", recursive=True)
            + glob.glob(f"./**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(
            f"{name} not found. On Kaggle, click 'Add Input' and add the "
            f"'Titanic - Machine Learning from Disaster' competition data. "
            f"Locally, put train.csv and test.csv next to this script."
        )
    return hits[0]


train = pd.read_csv(find_file("train.csv"))
test = pd.read_csv(find_file("test.csv"))
print("Train:", train.shape, "| Test:", test.shape)


# ---------- Feature engineering ----------
def add_features(df):
    df = df.copy()
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    rare = ["Dr", "Rev", "Col", "Major", "Capt", "Sir", "Lady", "Don",
            "Dona", "Jonkheer", "the Countess"]
    df["Title"] = df["Title"].replace(rare, "Rare")
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("U")
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
    return df


train = add_features(train)
test = add_features(test)

num_cols = ["Age", "Fare", "FarePerPerson", "SibSp", "Parch",
            "FamilySize", "IsAlone", "HasCabin", "Pclass"]
cat_cols = ["Sex", "Embarked", "Title", "Deck"]
features = num_cols + cat_cols

X = train[features]
y = train["Survived"]
X_test = test[features]

# ---------- Preprocessing ----------
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0),
    "RandomForest": RandomForestClassifier(
        n_estimators=500, max_depth=6, min_samples_leaf=3,
        random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
}

# ---------- Cross-validate each model ----------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_name, best_score = None, 0
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    score = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy").mean()
    print(f"{name}: CV accuracy = {score:.4f}")
    if score > best_score:
        best_name, best_score = name, score

print(f"\nBest model: {best_name} ({best_score:.4f})")

# ---------- Fit best model on full train, predict test ----------
final = Pipeline([("prep", preprocess), ("clf", models[best_name])])
final.fit(X, y)
preds = final.predict(X_test)

out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": preds})
submission.to_csv(f"{out_dir}/submission.csv", index=False)
print(f"Saved {out_dir}/submission.csv with {len(submission)} rows")

Train: (891, 12) | Test: (418, 11)
LogisticRegression: CV accuracy = 0.8260
RandomForest: CV accuracy = 0.8417
GradientBoosting: CV accuracy = 0.8350

Best model: RandomForest (0.8417)
Saved /kaggle/working/submission.csv with 418 rows
